# Task B -- final full-data fit with one-layer reinitialization

This notebook trains the current best observed Task B recipe on every labelled example and predicts the official validation inputs. The classifier uses `--reinit-layers 1`, preserving MuRIL's 11th layer and reinitializing only the final layer.

| setting | value |
|---|---|
| TAPT corpus | D0: Task B training comments + OffensEval Kannada |
| TAPT text | all 6,406 comments, no holdout, no filtering or deduplication |
| classifier | all 3,159 labelled Task B rows, no deduplication |
| reinitialization | **one** top encoder layer (`--reinit-layers 1`) |
| seeds | 42, 43, 44, 45, 46; probabilities averaged |
| score | CodaBench only; no local F1 exists for a full fit |

This setting is provisional until the seed-43/44 confirmation experiments finish. Expected runtime is about 2 hours on a T4. Upload this notebook with GPU and Internet enabled, then use **Save Version -> Save & Run All**.

In [ ]:
import os, pathlib, re, shutil, subprocess, sys

WORK = "/kaggle/working/hastika"
if os.path.isdir(WORK + "/.git"):
    subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", "task-b", "--depth", "1",
                    "https://github.com/robinpnalex/Hastika-ICON2026.git", WORK], check=True)
os.chdir(WORK)
os.environ["PYTHONPATH"] = os.path.join(WORK, "src")
sys.path.insert(0, os.path.join(WORK, "src"))
pathlib.Path("artifacts/logs").mkdir(parents=True, exist_ok=True)
print("repo:", os.getcwd())
subprocess.run(["git", "log", "-1", "--oneline"], check=True)
subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"',
               shell=True, check=True)
import torch
assert torch.cuda.is_available(), "no GPU -- set Accelerator in the sidebar"
print("gpu:", torch.cuda.get_device_name(0))

def run(cmd, log=None):
    print("$", " ".join(cmd), flush=True)
    fh = open(log, "w") if log else None
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        sys.stdout.write(line)
        if fh:
            fh.write(line)
    p.wait()
    if fh:
        fh.close()
    if p.returncode:
        raise RuntimeError(f"exit {p.returncode}: {cmd}")


## 1. TAPT on all permitted text

No labelled examples are held back at the adaptation stage. The corpus is restricted to the Task B training comments and the permitted external Kannada corpus; Task A and official validation inputs remain excluded.

In [ ]:
D0_CORPUS = ["data/raw/multiclass_train.csv", "data/external/offenseval_kn.csv"]
V0_MODEL = "google/muril-base-cased"
TAPT_OUT = "artifacts/runs/tapt-d0v0-reinit1-100"
TAPT_LOG = "artifacts/logs/tapt_d0v0_reinit1_100.log"

if pathlib.Path(TAPT_OUT).is_dir():
    print("using existing TAPT checkpoint:", TAPT_OUT)
else:
    run([sys.executable, "-u", "-m", "hastika.task_b.tapt",
         "--model", V0_MODEL, "--corpus", *D0_CORPUS, "--epochs", "8",
         "--val-frac", "0", "--min-words", "1", "--no-dedupe",
         "--out", TAPT_OUT], log=TAPT_LOG)
assert pathlib.Path(TAPT_OUT).is_dir(), "TAPT checkpoint was not written"

if pathlib.Path(TAPT_LOG).exists():
    t = pathlib.Path(TAPT_LOG).read_text()
    m = re.search(r"MLM trains on (\d+) of (\d+) comments, (\d+) held out", t)
    assert m, "could not find the corpus line in the TAPT log"
    used, total, held = map(int, m.groups())
    print(f"TAPT trained on {used} of {total} comments, {held} held out")
    assert used == total == 6406 and held == 0, "TAPT did not use all 6,406 comments"
print("TAPT checkpoint ready:", TAPT_OUT)


## 2. Five-seed full-data classifier fit

Each seed trains on all 3,159 labelled rows. There is no local F1 because the official validation labels are hidden; the next cell checks that every seed really ran on the full dataset.

In [ ]:
TAG = "b_reinit1_full"
TRAIN_LOG = f"artifacts/logs/{TAG}.log"
SEEDS = ["42", "43", "44", "45", "46"]
run([sys.executable, "-u", "-m", "hastika.task_b.train",
     "--tag", TAG, "--model", TAPT_OUT, "--folds", "1",
     "--no-dedupe", "--reinit-layers", "1", "--aux-weight", "0",
     "--seeds", *SEEDS, "--epochs", "6"], log=TRAIN_LOG)

t = pathlib.Path(TRAIN_LOG).read_text()
fits = re.findall(r"===== seed (\d+) FULL FIT, (\d+) rows, no validation =====", t)
print("full fits:", fits)
assert [s for s, _ in fits] == SEEDS, "not all five seeds ran"
assert all(r == "3159" for _, r in fits), "a seed trained on fewer than 3,159 rows"
assert "reinit=1" in t, "training log does not show one-layer reinitialization"


## 3. Package predictions for CodaBench

The predictions are generated for `data/raw/multiclass_validation_inputs.csv`. The submission helper checks the `id,label` schema and all six valid labels, then writes the ZIP with a bare `predictions.csv` at its top level.

In [ ]:
ZIP = "/kaggle/working/b_reinit1_full.zip"
PRED = f"artifacts/runs/{TAG}/predictions.csv"
run([sys.executable, "-m", "hastika.common.submission",
     "--task", "b", "--pred", PRED, "--out", ZIP])
assert pathlib.Path(ZIP).exists(), "submission ZIP was not written"
run(["unzip", "-l", ZIP])

pred = pd.read_csv(PRED)
prior = pd.read_csv("data/raw/multiclass_train.csv")["Hate Category"].value_counts(normalize=True)
dist = pd.DataFrame({"predicted %": 100 * pred["label"].value_counts(normalize=True),
                     "training %": 100 * prior}).fillna(0).round(1)
print("predictions:", len(pred))
print("\npredicted distribution vs training prior:")
print(dist.to_string())
print(f"\nlargest drift: {(dist['predicted %'] - dist['training %']).abs().max():.1f} points")

OUT = pathlib.Path("/kaggle/working/reinit1_full_outputs")
OUT.mkdir(parents=True, exist_ok=True)
for source in [PRED, TRAIN_LOG, TAPT_LOG]:
    if pathlib.Path(source).exists():
        shutil.copy2(source, OUT / pathlib.Path(source).name)
shutil.copy2(pathlib.Path("artifacts/runs") / TAG / "test_probs.npy", OUT / f"{TAG}_test_probs.npy")
shutil.copy2(ZIP, OUT / pathlib.Path(ZIP).name)
print("\nDownload the ZIP and logs from:", OUT)
print("Upload b_reinit1_full.zip to the Task B validation phase on CodaBench.")


## 4. After CodaBench scores the submission

Record the returned macro-F1 in `docs/EXPERIMENTS.md` and `submissions/README.md`. This full-data fit has no honest local F1; CodaBench is its evaluation. Interpret small differences against earlier submissions cautiously because the 395-row validation set is noisy.